## Manual Masking Workflow for Ground-Truth Segmentation

This notebook is used to create polygon-based binary masks for thermal TIFF images.

### Objective
- Build high-quality manual masks for U-Net training and evaluation.
- Save masks with the same folder hierarchy and filename stem as the source TIFF.

### Output format
- Input: thermal TIFF images from patient-organized folders.
- Output: PNG binary masks in `GroundTruth_Masks`.

### Annotation controls
- Left click: add polygon point.
- `S`: save current mask.
- `C`: clear polygon and redraw.
- `Q`: quit current session safely.

### 1) Environment setup

Install the required packages once in your active environment.

If packages are already installed, this cell can be skipped.

In [1]:
pip install opencv-python tifffile numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2) Import libraries

These imports handle:
- image I/O (`tifffile`, `cv2`)
- polygon rasterization (`cv2.fillPoly`)
- array operations (`numpy`)
- recursive file discovery and path mapping (`glob`, `os`)

In [2]:
import cv2
import numpy as np
import tifffile as tiff
import os
import glob

### 3) Configure paths and build annotation batch

This cell:
1. Scans all TIFF files under `organized_by_patient`.
2. Checks whether each TIFF already has a corresponding PNG mask.
3. Builds `target_batch` from only unmasked files.

Adjust `target_batch = unmasked_tiffs[:150]` to control how many images you annotate in one session.

In [3]:
# --- CONFIGURATION ---
INPUT_BASE = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\organized_by_patient"
OUTPUT_BASE = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\GroundTruth_Masks"

# 1. Find all TIFFs
all_tiffs = glob.glob(os.path.join(INPUT_BASE, "**", "*.tiff"), recursive=True)

# 2. Find which ones are NOT masked yet
unmasked_tiffs = []
for tiff_path in all_tiffs:
    rel_path = os.path.relpath(tiff_path, INPUT_BASE)
    mask_name = os.path.splitext(rel_path)[0] + ".png"
    mask_path = os.path.join(OUTPUT_BASE, mask_name)
    
    if not os.path.exists(mask_path):
        unmasked_tiffs.append(tiff_path)

print(f"Total TIFFs: {len(all_tiffs)}")
print(f"Already Masked: {len(all_tiffs) - len(unmasked_tiffs)}")
print(f"Remaining to Mask: {len(unmasked_tiffs)}")

# Choose how many you want to do in this session (e.g., 150)
target_batch = unmasked_tiffs[:150]

Total TIFFs: 657
Already Masked: 35
Remaining to Mask: 622


### 4) Interactive polygon annotation loop

This section launches the OpenCV annotation window and iterates through `target_batch`.

### What the script does per image
1. Loads TIFF and applies min-max normalization.
2. Resizes to 256x256 and colorizes for better visibility.
3. Lets you draw a closed polygon around the breast region.
4. Saves a binary PNG mask with matching relative path.

### Practical annotation guidance
- Exclude neck and upper torso background.
- Stop at lateral fold to avoid armpit leakage.
- Ensure the polygon traces the inframammary contour consistently across views.

In [4]:
pts = []
current_display_img = None

def mouse_callback(event, x, y, flags, param):
    global pts, current_display_img
    if event == cv2.EVENT_LBUTTONDOWN:
        pts.append((x, y))
        cv2.circle(current_display_img, (x, y), 3, (0, 0, 255), -1)
        if len(pts) > 1:
            cv2.line(current_display_img, pts[-2], pts[-1], (0, 255, 0), 1)
        cv2.imshow("V2 Stronger Masking", current_display_img)

def run_v2_annotator():
    global pts, current_display_img
    cv2.namedWindow("V2 Stronger Masking")
    cv2.setMouseCallback("V2 Stronger Masking", mouse_callback)

    print("--- INSTRUCTIONS FOR V2 ---")
    print("1. AVOID THE NECK: Stop at the top of the breast.")
    print("2. AVOID ARMPITS: Stop at the lateral fold.")
    print("3. 'S' to Save | 'C' to Clear | 'Q' to Quit")

    for file_path in target_batch:
        # Load raw TIFF
        try:
            raw_data = tiff.imread(file_path)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            continue

        # Normalization per Paper Eq 1
        t_min, t_max = np.min(raw_data), np.max(raw_data)
        normalized = 255 * (raw_data - t_min) / (t_max - t_min + 1e-8)
        img_256 = cv2.resize(normalized.astype(np.uint8), (256, 256), interpolation=cv2.INTER_AREA)
        
        # Colorize for visibility
        base_display = cv2.applyColorMap(img_256, cv2.COLORMAP_MAGMA)
        current_display_img = base_display.copy()
        pts = []

        print(f"Annotating: {os.path.basename(file_path)}")

        while True:
            cv2.imshow("V2 Stronger Masking", current_display_img)
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('s'):
                if len(pts) > 2:
                    mask = np.zeros((256, 256), dtype=np.uint8)
                    cv2.fillPoly(mask, [np.array(pts)], 255)
                    
                    rel_path = os.path.relpath(file_path, INPUT_BASE)
                    mask_name = os.path.splitext(rel_path)[0] + ".png"
                    save_path = os.path.join(OUTPUT_BASE, mask_name)
                    
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    cv2.imwrite(save_path, mask)
                    print(f"Saved: {mask_name}")
                    break
            elif key == ord('c'):
                current_display_img = base_display.copy()
                pts = []
            elif key == ord('q'):
                cv2.destroyAllWindows()
                return

    cv2.destroyAllWindows()
    print("Session Complete!")

run_v2_annotator()

--- INSTRUCTIONS FOR V2 ---
1. AVOID THE NECK: Stop at the top of the breast.
2. AVOID ARMPITS: Stop at the lateral fold.
3. 'S' to Save | 'C' to Clear | 'Q' to Quit
Annotating: Left Lateral (90°).tiff
